In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
# %load_ext cudf.pandas

In [3]:
# STEFANOS: Disable pip install
# !pip install lazypredict
# !pip install --upgrade pandas
# !pip install fast-tabnet
# !pip install fastai
# !pip install pandas-profiling

In [4]:
#A program that takes a csv and trains models on it. Streamlined model selection.
#==============================================================================

# STEFANOS: Disable unneeded modules
# #LazyPredict
# import lazypredict
# from lazypredict.Supervised import LazyRegressor
# from lazypredict.Supervised import LazyClassifier
# #Baysian Optimization
# from bayes_opt import BayesianOptimization
#Pandas stack
import os
# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
# import pandas_profiling
import numpy as np
# STEFANOS: Disable unneeded modules
# #FastAI
from fastai.tabular.all import *
from fastai.tabular.core import *
# #Plots
# import matplotlib.pyplot as plt
# import seaborn as sns
#System
import os
import sys
import traceback
#Fit an xgboost model
# STEFANOS: Disable unneeded modules
# from xgboost import XGBRegressor
# from xgboost import XGBClassifier
# from xgboost import plot_importance
# from sklearn.metrics import mean_squared_error
# from sklearn.metrics import roc_auc_score
#Random
import random

#TabNet
from fast_tabnet.core import *

import shutil

In [ ]:
import time
import gc
start_time = time.time()

In [ ]:
# STEFANOS: Disable matplotlib inline
# %matplotlib inline

In [ ]:
# For Styling
# STEFANOS: Disbale plotting
# plt.style.use('seaborn-bright')

In [5]:
#Project Variables
#===================================================================================================
PROJECT_NAME = 'superstore'
VARIABLE_FILES = False
#Maximum amount of rows to take

## -- STEFANOS -- Remove sample

# SAMPLE_COUNT = 4000
FASTAI_LEARNING_RATE = 1e-1
AUTO_ADJUST_LEARNING_RATE = False
#Set to True automatically infer if variables are categorical or continuous
ENABLE_BREAKPOINT = True
#When trying to declare a column a continuous variable, if it fails, convert it to a categorical variable
CONVERT_TO_CAT = False
REGRESSOR = True
SEP_DOLLAR = False
SEP_COMMA = True
SHUFFLE_DATA = True

In [6]:
### cell 0 ###
input_dir = f'../input/{PROJECT_NAME}'
# STEFANOS: Change the working path
# param_dir = f'/kaggle/working/{PROJECT_NAME}'
param_dir = f'./working/{PROJECT_NAME}'
TARGET = ''
PARAM_DIR = param_dir
print(f'param_dir: {param_dir}')
if not os.path.exists(param_dir):
    os.makedirs(param_dir)
#rename any file in param_dir/file that ends with csv to data.csv
for file in os.listdir(input_dir):
    if file.endswith('.csv'):
        print('CSV!')
        if 'classification_results' not in file and 'regression_results' not in file:
            #os.rename(f'{input_dir}/{file}', f'{param_dir}/data.csv')
            shutil.copy(f'{input_dir}/{file}', f'{param_dir}/data.csv')
        #os.rename(f'{param_dir}/{file}', f'{param_dir}/data.csv')
try:
    # STEFANOS: Remove sample
#     df = pd.read_csv(f'{param_dir}/data.csv', nrows=SAMPLE_COUNT)
    df = pd.read_csv(f'{param_dir}/data.csv')
except:
    print(f'Please place a file named data.csv in {param_dir}')

param_dir: ./working/superstore
CSV!


In [ ]:
%%timeit
### cell 1 ###
df

In [ ]:
%%cudf.pandas.profile
df

# -- STEFANOS -- Replicate Data

In [ ]:
%%cudf.pandas.profile
%%timeit
### cell 2 ###
factor = 10
df_expanded = pd.concat([df]*factor)
df_expanded.info()
del df_expanded

In [7]:
factor = 10
df = pd.concat([df]*factor)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 99940 entries, 0 to 9993
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Ship Mode     99940 non-null  object 
 1   Segment       99940 non-null  object 
 2   Country       99940 non-null  object 
 3   City          99940 non-null  object 
 4   State         99940 non-null  object 
 5   Postal Code   99940 non-null  int64  
 6   Region        99940 non-null  object 
 7   Category      99940 non-null  object 
 8   Sub-Category  99940 non-null  object 
 9   Sales         99940 non-null  float64
 10  Quantity      99940 non-null  int64  
 11  Discount      99940 non-null  float64
 12  Profit        99940 non-null  float64
dtypes: float64(3), int64(2), object(8)
memory usage: 10.7+ MB


In [ ]:
df_copy = df.copy()

In [ ]:
%%timeit
### cell 3 ###
if SEP_DOLLAR:
    #For every column in df_copy, if the column contains a $, make a new column with the value without the $
    for col in df_copy.columns:
        if '$' in df_copy[col].to_string():
            df_copy[col + '_no_dollar'] = df_copy[col].str.replace('$', '').str.replace(',', '')
            #Try to convert this new column to a numeric type
            try:
                df_copy[col + '_no_dollar'] = df_copy[col + '_no_dollar'].apply(pd.to_numeric, errors='coerce').dropna()
            except Exception:
                print(f'{col} can not be converted to a float!')


if SEP_COMMA:
    #For every column in df_copy, if the column contains a %, make a new column with the value without the %
    for col in df_copy.columns:
        if '%' in df_copy[col].to_string() or ',' in df_copy[col].to_string():
            df_copy[col + '_processed'] = df_copy[col].apply(str).str.replace('%', '').str.replace(',', '')
            #Try to convert this new column to a numeric type
            try:
                df_copy[col + '_processed'] = df_copy[col + '_processed'].apply(pd.to_numeric, errors='coerce').dropna()
            except Exception:
                print(f'{col} can not be converted to a float!')

In [ ]:
df_copy = df.copy()

In [ ]:
%%timeit
# Rewrite first loop
for col in df.columns:
    # Check if the column contains a '$'
    try:
        if df_copy[col].str.contains(r'\$').any():
            # Remove '$' and commas
            new_col = df_copy[col].str.replace(r'\$', '', regex=True).str.replace(r',', '', regex=True)
            try:
                # Convert to float and drop NaN values
                new_col = new_col.astype('float64').dropna()
            except Exception:
                print(f"{col} cannot be converted to float!")
            df_copy[col + '_no_dollar'] = new_col
    except Exception:
        # Skip columns that do not support string operations
        continue

In [ ]:
del df_copy

In [8]:
if SEP_DOLLAR:
    #For every column in df_copy, if the column contains a $, make a new column with the value without the $
    for col in df.columns:
        if '$' in df[col].to_string():
            df[col + '_no_dollar'] = df[col].str.replace('$', '').str.replace(',', '')
            #Try to convert this new column to a numeric type
            try:
                df[col + '_no_dollar'] = df[col + '_no_dollar'].apply(pd.to_numeric, errors='coerce').dropna()
            except Exception:
                print(f'{col} can not be converted to a float!')


if SEP_COMMA:
    #For every column in df_copy, if the column contains a %, make a new column with the value without the %
    for col in df.columns:
        if '%' in df[col].to_string() or ',' in df[col].to_string():
            df[col + '_processed'] = df[col].apply(str).str.replace('%', '').str.replace(',', '')
            #Try to convert this new column to a numeric type
            try:
                df[col + '_processed'] = df[col + '_processed'].apply(pd.to_numeric, errors='coerce').dropna()
            except Exception:
                print(f'{col} can not be converted to a float!')

In [ ]:
%%timeit
### cell 4 ###
df

In [ ]:
%%cudf.pandas.profile
df

In [ ]:
%%memit
df

In [ ]:
%%timeit
### cell 5 ###
df.isna().sum()

In [ ]:
%%cudf.pandas.profile
df.isna().sum()

In [ ]:
%%memit
df.isna().sum()

In [ ]:
%%timeit
### cell 6 ###
df.profile_report()

In [ ]:
%%timeit
### cell 7 ###
# STEFANOS: Disable plotting
# sns.heatmap(df.corr())
df.select_dtypes(include=['number']).corr()

In [ ]:
%%cudf.pandas.profile
df.select_dtypes(include=['number']).corr()

In [ ]:
%%memit
df.select_dtypes(include=['number']).corr()

In [ ]:
%%timeit
### cell 8 ###
# STEFANOS: Disable plotting
# df.head().style.background_gradient(cmap = "inferno")
df.head()

In [ ]:
%%cudf.pandas.profile
df.head()

In [ ]:
%%memit
df.head()

In [ ]:
%%timeit
### cell 9 ###
# STEFANOS: Disable plotting
# df.describe().T.style.background_gradient(cmap = "viridis")
df.describe().T

In [ ]:
%%cudf.pandas.profile
df.describe().T

In [ ]:
%%memit
df.describe().T

In [ ]:
%%timeit
### cell 10 ###
df.columns

In [ ]:
%%cudf.pandas.profile
df.columns

In [ ]:
%%memit
df.columns

In [9]:
df_new = df.copy()
df_original = df.copy()

In [10]:
# TODO(jie) v2
start_time = time.time()
target = ''
target_str = ''
#The column closest to the end isPARAM_DIR the target variable that can be represented as a float is the target variable
targets = []
#Loop through every possible target column (Continuous)
for i in range(len(df_new.columns)-1, 0, -1):
    target = df_new.columns[i]
    df_new[target] = pd.to_numeric(df_new[target], errors='coerce').dropna()
    #Will be determined by the file name

    #===================================================================================================

    #Create project config files if they don't exist.
    if not os.path.exists(param_dir):
        #create param_dir
        os.makedirs(PARAM_DIR)
    if not os.path.exists(f'{PARAM_DIR}/cats.txt'):
        #create param_dir
        with open(f'{PARAM_DIR}/cats.txt', 'w') as f:
            f.write('')
    if not os.path.exists(f'{PARAM_DIR}/conts.txt'):
        #create param_dir
        with open(f'{PARAM_DIR}/conts.txt', 'w') as f:
            f.write('')
    if not os.path.exists(f'{PARAM_DIR}/cols_to_delete.txt'):
        with open(f'{PARAM_DIR}/cols_to_delete.txt', 'w') as f:
            f.write('')

    df_new = df_new.drop_duplicates()
    if SHUFFLE_DATA:
        df_new = df_new.sample(frac=1).reset_index(drop=True)

    # workaround for fastai/pytorch bug where bool is treated as object and thus erroring out.
    bool_cols = [col for col in df_new.columns if df_new[col].dtype == 'bool']
    if bool_cols:
        df_new[bool_cols] = df_new[bool_cols].astype('uint8')

    with open(f'{PARAM_DIR}/cols_to_delete.txt', 'r') as f:
        cols_to_delete = f.read().splitlines()
    for col in cols_to_delete:
        try:
            del(df_new[col])
        except:
            pass
    #try to fill in missing values now, otherwise FastAI will do it for us later
    try:
        df_new = df_new.fillna(0)
    except:
        pass
 
    df_new = df_shrink(df_new)

    #Auto detect categorical and continuous variables
    #==============================================================================
    # Auto detect categorical and continuous variables using vectorized operations
    unique_counts = df_new.nunique()      # Unique count for each column
    non_null_counts = df_new.count()        # Non-null count for each column
    ratio = unique_counts / non_null_counts
    
    # Identify likely categorical columns (ratio < threshold)
    likely_cat = ratio < 0.05
    
    # Get lists of categorical and continuous columns
    cats = list(likely_cat[likely_cat].index)
    conts = list(likely_cat[~likely_cat].index)
    
    # Remove the target column from both lists if present
    for lst in (cats, conts):
        try:
            lst.remove(target)
        except ValueError:
            pass
    #Convert target to float
    df_new[target] = pd.to_numeric(df_new[target], errors='coerce').dropna()

    #Populate categorical and continuous lists
    #==============================================================================

    if VARIABLE_FILES == True:
        with open(f'{PARAM_DIR}/cats.txt', 'r') as f:
            cats = f.read().splitlines()

        with open(f'{PARAM_DIR}/conts.txt', 'r') as f:
            conts = f.read().splitlines()

    #==============================================================================

    #==============================================================================
    procs = [Categorify, FillMissing, Normalize]
    #print(df_new.describe().T)
    # STEFANOS: Remove samples
#     df_new = df_new[0:SAMPLE_COUNT]
    splits = RandomSplitter()(range_of(df_new))

    #conts = []

    #Convert cont variables to floats
    #==============================================================================

    #Convert cont variables to floats
    #==============================================================================

    for var in conts:
        try:
            df_new[var] = pd.to_numeric(df_new[var], errors='coerce').dropna()
        except:
            pass

    #==============================================================================

    #Experimental logic to add columns one-by-one to find a breakpoint
    #==============================================================================
    if ENABLE_BREAKPOINT == True:
        temp_procs = [Categorify, FillMissing]
        print('Looping through continuous variables to find breakpoint')
        cont_list = []
        for cont in conts:
            cont_list.append(cont)
            #print(focus_cont)
            try:
                to = TabularPandas(
                    df_new, 
                    procs=procs, 
                    cat_names=cats, 
                    cont_names=cont_list, 
                    y_names=target, 
                    y_block=RegressionBlock(), 
                    splits=splits
                )
                del(to)
            except:
                #remove focus_cont from list
                cont_list.remove(focus_cont)
                #traceback.print_exc()
                continue
        #convert all continuous variables to floats
        for var in cont_list:
            try:
                df_new[var] = pd.to_numeric(df_new[var], errors='coerce').dropna()
            except:
                cont_list.remove(var)
                if CONVERT_TO_CAT == True:
                    cats.append(var)
                pass
        #shrink df_new as much as possible
        df_new = df_shrink(df_new)
        #print(df_new.dtypes)

    #==============================================================================

    #Creating tabular object + quick preprocessing
    #==============================================================================
    to = None
    if REGRESSOR == True:
        try:
            to = TabularPandas(df_new, procs, cats, conts, target, y_block=RegressionBlock(), splits=splits)
        except:
            conts = []
            to = TabularPandas(df_new, procs, cats, conts, target, y_block=RegressionBlock(), splits=splits)
    else:
        try:
            to = TabularPandas(df_new, procs, cats, conts, target, splits=splits)
        except:
            conts = []
            to = TabularPandas(df_new, procs, cats, conts, target, splits=splits)
end_time = time.time()
print(end_time - start_time)

Looping through continuous variables to find breakpoint
Looping through continuous variables to find breakpoint
Looping through continuous variables to find breakpoint
Looping through continuous variables to find breakpoint
Looping through continuous variables to find breakpoint
Looping through continuous variables to find breakpoint
Looping through continuous variables to find breakpoint
Looping through continuous variables to find breakpoint
Looping through continuous variables to find breakpoint
Looping through continuous variables to find breakpoint
Looping through continuous variables to find breakpoint
Looping through continuous variables to find breakpoint
1.9435336589813232


In [13]:
df = df_original.copy()

In [14]:
# TODO(sahil)
target = ''
target_str = ''
#The column closest to the end isPARAM_DIR the target variable that can be represented as a float is the target variable
targets = []
#Loop through every possible target column (Continuous)
for i in range(len(df.columns)-1, 0, -1):
    try:
        df[df.columns[i]] = df[df.columns[i]].apply(pd.to_numeric, errors='coerce').dropna()
        target = df.columns[i]
        target_str = target.replace('/', '-')
    except:
        continue
    print(f'Target Variable: {target}')
    #Will be determined by the file name


    #===================================================================================================

    #Create project config files if they don't exist.
    if not os.path.exists(param_dir):
        #create param_dir
        os.makedirs(PARAM_DIR)
    if not os.path.exists(f'{PARAM_DIR}/cats.txt'):
        #create param_dir
        with open(f'{PARAM_DIR}/cats.txt', 'w') as f:
            f.write('')
    if not os.path.exists(f'{PARAM_DIR}/conts.txt'):
        #create param_dir
        with open(f'{PARAM_DIR}/conts.txt', 'w') as f:
            f.write('')
    if not os.path.exists(f'{PARAM_DIR}/cols_to_delete.txt'):
        with open(f'{PARAM_DIR}/cols_to_delete.txt', 'w') as f:
            f.write('')

    df = df.drop_duplicates()
    if SHUFFLE_DATA:
        df = df.sample(frac=1).reset_index(drop=True)

    # workaround for fastai/pytorch bug where bool is treated as object and thus erroring out.
    for n in df:
        if pd.api.types.is_bool_dtype(df[n]):
            df[n] = df[n].astype('uint8')

    with open(f'{PARAM_DIR}/cols_to_delete.txt', 'r') as f:
        cols_to_delete = f.read().splitlines()
    for col in cols_to_delete:
        try:
            del(df[col])
        except:
            pass
    #try to fill in missing values now, otherwise FastAI will do it for us later
    try:
        df = df.fillna(0)
    except:
        pass
    #print missing values
    #print(df.isna().sum().sort_values(ascending=False))
    #shrink df as much as possible
    df = df_shrink(df)


    #print types inside of df
    #print(df.dtypes)


    #Auto detect categorical and continuous variables
    #==============================================================================
    likely_cat = {}
    for var in df.columns:
        likely_cat[var] = 1.*df[var].nunique()/df[var].count() < 0.05 #or some other threshold

    cats = [var for var in df.columns if likely_cat[var]]
    conts = [var for var in df.columns if not likely_cat[var]]

    #remove target from lists
    try:
        conts.remove(target)
        cats.remove(target)
    except:
        pass
    #Convert target to float
    df[target] = df[target].apply(pd.to_numeric, errors='coerce').dropna()

    print('CATS=====================')
    print(cats)
    print('CONTS=====================')
    print(conts)

    # TODO(sahil): Continue from below

    #Populate categorical and continuous lists
    #==============================================================================

    if VARIABLE_FILES == True:
        with open(f'{PARAM_DIR}/cats.txt', 'r') as f:
            cats = f.read().splitlines()

        with open(f'{PARAM_DIR}/conts.txt', 'r') as f:
            conts = f.read().splitlines()

    #==============================================================================

    #==============================================================================
    procs = [Categorify, FillMissing, Normalize]
    #print(df.describe().T)
    # STEFANOS: Remove samples
#     df = df[0:SAMPLE_COUNT]
    splits = RandomSplitter()(range_of(df))

    print((len(cats)) + len(conts))
    #conts = []

    #Convert cont variables to floats
    #==============================================================================

    #Convert cont variables to floats
    #==============================================================================

    for var in conts:
        try:
            df[var] = df[var].apply(pd.to_numeric, errors='coerce').dropna()
        except:
            print(f'Could not convert {var} to float.')
            pass

    #==============================================================================

    #Experimental logic to add columns one-by-one to find a breakpoint
    #==============================================================================
    if ENABLE_BREAKPOINT == True:
        temp_procs = [Categorify, FillMissing]
        print('Looping through continuous variables to find breakpoint')
        cont_list = []
        for cont in conts:
            focus_cont = cont
            cont_list.append(cont)
            #print(focus_cont)
            try:
                to = TabularPandas(df, procs=procs, cat_names=cats, cont_names=cont_list, y_names=target, y_block=RegressionBlock(), splits=splits)
                del(to)
            except:
                print('Error with ', focus_cont)
                #remove focus_cont from list
                cont_list.remove(focus_cont)
                #traceback.print_exc()
                continue
        #convert all continuous variables to floats
        for var in cont_list:
            try:
                df[var] = df[var].apply(pd.to_numeric, errors='coerce').dropna()
            except:
                print(f'Could not convert {var} to float.')
                cont_list.remove(var)
                if CONVERT_TO_CAT == True:
                    cats.append(var)
                pass
        print(f'Continuous variables that made the cut : {cont_list}')
        print(f'Categorical variables that made the cut : {cats}')
        #shrink df as much as possible
        df = df_shrink(df)
        #print(df.dtypes)

    #==============================================================================

    #Creating tabular object + quick preprocessing
    #==============================================================================
    to = None
    if REGRESSOR == True:
        try:
            to = TabularPandas(df, procs, cats, conts, target, y_block=RegressionBlock(), splits=splits)
        except:
            conts = []
            to = TabularPandas(df, procs, cats, conts, target, y_block=RegressionBlock(), splits=splits)
    else:
        try:
            to = TabularPandas(df, procs, cats, conts, target, splits=splits)
        except:
            conts = []
            to = TabularPandas(df, procs, cats, conts, target, splits=splits)

Target Variable: Profit
CATS=====================
['Ship Mode', 'Segment', 'Country', 'State', 'Region', 'Category', 'Sub-Category', 'Quantity', 'Discount']
CONTS=====================
['City', 'Postal Code', 'Sales']
12
Looping through continuous variables to find breakpoint
Continuous variables that made the cut : ['City', 'Postal Code', 'Sales']
Categorical variables that made the cut : ['Ship Mode', 'Segment', 'Country', 'State', 'Region', 'Category', 'Sub-Category', 'Quantity', 'Discount', 'City_na']
Target Variable: Discount
CATS=====================
['Ship Mode', 'Segment', 'Country', 'State', 'Region', 'Category', 'Sub-Category', 'Quantity', 'Discount']
CONTS=====================
['City', 'Postal Code', 'Sales', 'Profit']
13
Looping through continuous variables to find breakpoint
Continuous variables that made the cut : ['City', 'Postal Code', 'Sales', 'Profit']
Categorical variables that made the cut : ['Ship Mode', 'Segment', 'Country', 'State', 'Region', 'Category', 'Sub-Cate

In [ ]:
%%memit
target = ''
target_str = ''
#The column closest to the end isPARAM_DIR the target variable that can be represented as a float is the target variable
targets = []
#Loop through every possible target column (Continuous)
for i in range(len(df.columns)-1, 0, -1):
    try:
        df[df.columns[i]] = df[df.columns[i]].apply(pd.to_numeric, errors='coerce').dropna()
        target = df.columns[i]
        target_str = target.replace('/', '-')
    except:
        continue
    print(f'Target Variable: {target}')
    #Will be determined by the file name


    #===================================================================================================

    #Create project config files if they don't exist.
    if not os.path.exists(param_dir):
        #create param_dir
        os.makedirs(PARAM_DIR)
    if not os.path.exists(f'{PARAM_DIR}/cats.txt'):
        #create param_dir
        with open(f'{PARAM_DIR}/cats.txt', 'w') as f:
            f.write('')
    if not os.path.exists(f'{PARAM_DIR}/conts.txt'):
        #create param_dir
        with open(f'{PARAM_DIR}/conts.txt', 'w') as f:
            f.write('')
    if not os.path.exists(f'{PARAM_DIR}/cols_to_delete.txt'):
        with open(f'{PARAM_DIR}/cols_to_delete.txt', 'w') as f:
            f.write('')

    df = df.drop_duplicates()
    if SHUFFLE_DATA:
        df = df.sample(frac=1).reset_index(drop=True)

    # workaround for fastai/pytorch bug where bool is treated as object and thus erroring out.
    for n in df:
        if pd.api.types.is_bool_dtype(df[n]):
            df[n] = df[n].astype('uint8')

    with open(f'{PARAM_DIR}/cols_to_delete.txt', 'r') as f:
        cols_to_delete = f.read().splitlines()
    for col in cols_to_delete:
        try:
            del(df[col])
        except:
            pass
    #try to fill in missing values now, otherwise FastAI will do it for us later
    try:
        df = df.fillna(0)
    except:
        pass
    #print missing values
    #print(df.isna().sum().sort_values(ascending=False))
    #shrink df as much as possible
    df = df_shrink(df)


    #print types inside of df
    #print(df.dtypes)


    #Auto detect categorical and continuous variables
    #==============================================================================
    likely_cat = {}
    for var in df.columns:
        likely_cat[var] = 1.*df[var].nunique()/df[var].count() < 0.05 #or some other threshold

    cats = [var for var in df.columns if likely_cat[var]]
    conts = [var for var in df.columns if not likely_cat[var]]

    #remove target from lists
    try:
        conts.remove(target)
        cats.remove(target)
    except:
        pass
    #Convert target to float
    df[target] = df[target].apply(pd.to_numeric, errors='coerce').dropna()

    print('CATS=====================')
    print(cats)
    print('CONTS=====================')
    print(conts)

    #Populate categorical and continuous lists
    #==============================================================================

    if VARIABLE_FILES == True:
        with open(f'{PARAM_DIR}/cats.txt', 'r') as f:
            cats = f.read().splitlines()

        with open(f'{PARAM_DIR}/conts.txt', 'r') as f:
            conts = f.read().splitlines()

    #==============================================================================

    #==============================================================================
    procs = [Categorify, FillMissing, Normalize]
    #print(df.describe().T)
    # STEFANOS: Remove samples
#     df = df[0:SAMPLE_COUNT]
    splits = RandomSplitter()(range_of(df))

    print((len(cats)) + len(conts))
    #conts = []

    #Convert cont variables to floats
    #==============================================================================

    #Convert cont variables to floats
    #==============================================================================

    for var in conts:
        try:
            df[var] = df[var].apply(pd.to_numeric, errors='coerce').dropna()
        except:
            print(f'Could not convert {var} to float.')
            pass

    #==============================================================================

    #Experimental logic to add columns one-by-one to find a breakpoint
    #==============================================================================
    if ENABLE_BREAKPOINT == True:
        temp_procs = [Categorify, FillMissing]
        print('Looping through continuous variables to find breakpoint')
        cont_list = []
        for cont in conts:
            focus_cont = cont
            cont_list.append(cont)
            #print(focus_cont)
            try:
                to = TabularPandas(df, procs=procs, cat_names=cats, cont_names=cont_list, y_names=target, y_block=RegressionBlock(), splits=splits)
                del(to)
            except:
                print('Error with ', focus_cont)
                #remove focus_cont from list
                cont_list.remove(focus_cont)
                #traceback.print_exc()
                continue
        #convert all continuous variables to floats
        for var in cont_list:
            try:
                df[var] = df[var].apply(pd.to_numeric, errors='coerce').dropna()
            except:
                print(f'Could not convert {var} to float.')
                cont_list.remove(var)
                if CONVERT_TO_CAT == True:
                    cats.append(var)
                pass
        print(f'Continuous variables that made the cut : {cont_list}')
        print(f'Categorical variables that made the cut : {cats}')
        #shrink df as much as possible
        df = df_shrink(df)
        #print(df.dtypes)

    #==============================================================================

    #Creating tabular object + quick preprocessing
    #==============================================================================
    to = None
    if REGRESSOR == True:
        try:
            to = TabularPandas(df, procs, cats, conts, target, y_block=RegressionBlock(), splits=splits)
        except:
            conts = []
            to = TabularPandas(df, procs, cats, conts, target, y_block=RegressionBlock(), splits=splits)
    else:
        try:
            to = TabularPandas(df, procs, cats, conts, target, splits=splits)
        except:
            conts = []
            to = TabularPandas(df, procs, cats, conts, target, splits=splits)

# -- STEFANOS -- We have to disable the following too because even though it's Pandas code, it uses results from ML

In [ ]:
# out_dir = f'./{PROJECT_NAME}'
# xgb_feature_importance_csvs = []

# for file in os.listdir(out_dir):
#     if 'xgb_feature_importance' in file and '.csv' in file:
#         xgb_feature_importance_csvs.append(pd.read_csv(os.path.join(out_dir, file)))

# xgb_feature_importance = pd.concat(xgb_feature_importance_csvs,axis=0)
# xgb_feature_importance.rename(columns={'Unnamed: 0': 'feature'}, inplace=True)
# print(xgb_feature_importance.head())
# xgb_feature_importance.groupby('feature')['importance'].mean().sort_values(ascending=False).plot(kind='bar', title='XGBoost Overall Feature Importance', figsize=(20, 10))

In [ ]:
%%timeit
### cell 12 ###
df.isna().sum()

In [ ]:
%%cudf.pandas.profile
df.isna().sum()

In [ ]:
end_time = time.time()

In [ ]:
print("Total time in seconds", end_time - start_time)

# **To Be Continued...**